#### <span style="color:#FFA726">Matrix Factorization & Alternating Least Squares</span>

This notebook has three parts:

1. **Pre-Code Notes** — the plan, columns used, and sanity checks, before any code.
2. **ALS Code** — building the interaction matrix, fitting the model, generating recommendations.
3. **Post-Code Sanity Checks** — verifying the model behaves as expected.

##### <span style="color:#FFA726">What are MF & ALS?</span>

<span style="color:#FFA726">**MF (Matrix Factorization):**</span> 

Breaks the huge, sparse user × item matrix into two small matrices:

**user matrix** (a hidden "taste vector" per user) and **item matrix** (a hidden "feature vector" per item)
 
Multiplying a user's vector by an item's vector (dot product) gives the predicted score of how much that user 
would like that item.

<span style="color:#FFA726">**ALS (Alternating Least Squares):**</span>

The method used to learn these two sets of vectors. It alternates, first fixing the user vectors and optimizing the item vectors, then the reverse, going back and forth until the vectors converge.

 **MF = the idea/model**

 **ALS = the method used to train it**.

#### <span style="color:#FFA726">1. Pre-Code Notes</span>

**Goal:** For each user, learn a latent vector, and for each item, learn a latent 
vector, such that the dot product of a user's vector and an item's vector predicts 
how much that user would like that item. Recommend the top-10 items with the 
highest predicted scores that the user hasn't already rated.

**Columns involved:**
- Used for building the interaction matrix: `user_id`, `parent_asin`
- `rating` is *not* used as a weighted value — this is implicit feedback: 

  **1** = that user rated that item
  
  **0** (empty) = they didn't


**Steps:**

1. **Build the interaction matrix:** using only `train`, build a sparse user × item 
   matrix where each entry is 1 if that user rated that item, 0 otherwise.
2. **Fit ALS:** train the ALS model on this matrix — it learns a latent vector for 
   every user and every item.
3. **Score:** for a given user, compute the dot product of their vector with every 
   item's vector to get a predicted score for every item.
4. **Filter and recommend:** exclude items already rated in `train`, take the top-10 
   remaining items by score.
5. **Sanity check:** for a handful of users, manually inspect whether their top-10 
   recommendations look reasonable given what they rated highly in `train`.


<span style="color:#66BB6A">**Note on implicit feedback:** 
we use rating as a binary signal (rated = 1, 
not rated = 0) rather than the star value itself. 
This means a 1-star and a 5-star 
rating are treated identically, both signal "the user interacted with this item.
This is a reasonable simplification here because our ratings are heavily skewed 
positive (62.6% are 5-star), so star value carries limited discriminative signal, 
and low ratings are rare and noisy. This binary approach is also standard practice 
for implicit-feedback recommenders like ALS.</span>

#### <span style="color:#FFA726">2. Fitting ALS</span>

In [3]:
!pip install implicit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 644.1/644.1 kB 14.5 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [5]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares

train = pd.read_parquet('datasets/train.parquet')
validation = pd.read_parquet('datasets/validation.parquet')
test = pd.read_parquet('datasets/test.parquet')

print("train:", len(train))
print("validation:", len(validation))
print("test:", len(test))

train: 6126723
validation: 657203
test: 657203


In [6]:
user_cat = train['user_id'].astype('category')
item_cat = train['parent_asin'].astype('category')

user_ids = user_cat.cat.codes.values
item_ids = item_cat.cat.codes.values

n_users = user_cat.cat.categories.size
n_items = item_cat.cat.categories.size

interaction_matrix = csr_matrix(
    (np.ones(len(train)), (user_ids, item_ids)),
    shape=(n_users, n_items)
)

print("Matrix shape:", interaction_matrix.shape)
print("Non-zero entries:", interaction_matrix.nnz)

Matrix shape: (657203, 197747)
Non-zero entries: 6126723


**Typical ranges for ALS parameters:**

| Parameter | Typical range | We used |
|---|---|---|
| `factors` | 20–200 | 50 |
| `iterations` | 10–20 | 15 |

These are starting values, not final ones, we'll tune them later on `validation` 
if the results look weak (per the project plan).

**What happens if you change them:**

| Parameter | Increase | Decrease |
|---|---|---|
| `factors` | Model captures more detail, but slower and risks overfitting (especially with our sparse users) | Faster, more general, but may miss patterns |
| `iterations` | Model converges more, but extra rounds beyond a point barely help | Faster, but model may not fully settle (underfit) |

In [8]:
model = AlternatingLeastSquares(factors=50, iterations=15, random_state=42)
model.fit(interaction_matrix)

print("Model fit completed ✅")

  0%|          | 0/15 [00:00<?, ?it/s]

Model fit completed ✅


In [9]:
sample_user_idx = 0  # index 0 in the interaction matrix

recommended_items, scores = model.recommend(
    sample_user_idx,
    interaction_matrix[sample_user_idx],
    N=10
)

recommended_asins = item_cat.cat.categories[recommended_items]

print("Sample user index:", sample_user_idx)
print("Recommended items:", recommended_asins.tolist())
print("Scores:", scores)

Sample user index: 0
Recommended items: ['B018FK66TU', 'B01DEBC7Q6', 'B00N1JQ2UO', 'B01BLS9E2Y', 'B00OV3VGP0', 'B008JFUPK8', 'B00AZMFK3K', 'B00G5G7K7O', 'B00K7IPGS6', 'B01IS31U6S']
Scores: [0.01874158 0.01059337 0.00993556 0.00985891 0.00963809 0.00878526
 0.00847606 0.00815093 0.00812453 0.00771444]


In [10]:
all_user_indices = np.arange(n_users)

all_recommended_items, all_scores = model.recommend(
    all_user_indices,
    interaction_matrix[all_user_indices],
    N=10
)

print("Shape of recommendations:", all_recommended_items.shape)

Shape of recommendations: (657203, 10)


#### <span style="color:#FFA726">3. Post-Code Sanity Checks</span>

We verify that the recommendations behave as expected: every user got exactly 
10 recommendations, and a manual check on a few users shows the recommended 
items are distinct from what they've already rated.

In [11]:
# 1. Confirm shape matches expected user count
print("Shape check:", all_recommended_items.shape == (n_users, 10))

# 2. Confirm every user got exactly 10 recommendations
rec_lengths = [len(row) for row in all_recommended_items]
print("Users with fewer than 10 recommendations:", sum(1 for l in rec_lengths if l < 10))
print("Users with exactly 10 recommendations:", sum(1 for l in rec_lengths if l == 10))

# 3. Manual check for a few users: compare recommendations against what they rated highly in train
for user_idx in [0, 1, 2]:
    user_id = user_cat.cat.categories[user_idx]
    rated_items = train[train['user_id'] == user_id][['parent_asin', 'rating']].sort_values('rating', ascending=False)
    recommended_asins = item_cat.cat.categories[all_recommended_items[user_idx]]
    
    print(f"\nUser {user_id}")
    print("Top rated in train:", rated_items.head(3).values.tolist())
    print("Recommended:", recommended_asins.tolist())

Shape check: True
Users with fewer than 10 recommendations: 0
Users with exactly 10 recommendations: 657203

User AE2222FRPDMNOMYOMCWIANTXP7UQ
Top rated in train: [['B0091X4AP8', 5.0], ['B0094K20FK', 5.0], ['B00BEIYHT2', 5.0]]
Recommended: ['B018FK66TU', 'B01DEBC7Q6', 'B00N1JQ2UO', 'B01BLS9E2Y', 'B00OV3VGP0', 'B008JFUPK8', 'B00AZMFK3K', 'B00G5G7K7O', 'B00K7IPGS6', 'B01IS31U6S']

User AE22236AFRRSMQIKGG7TPTB75QEA
Top rated in train: [['B0006B2A2O', 5.0], ['B001KZIRM2', 5.0], ['B000Y5JFNE', 5.0]]
Recommended: ['B018FK66TU', 'B00005JM5E', 'B01DEBC7Q6', 'B008JFUPK8', 'B00AZMFK3K', 'B00G5G7EXY', 'B00OV3VGP0', 'B00003CX5P', 'B00N1JQ452', 'B00N1JQ2UO']

User AE222H3FGXWLHRFUMGMS2RR57NDQ
Top rated in train: [['B009LDD1X0', 5.0], ['B0776JGNYG', 5.0], ['B01LTHMWRG', 5.0]]
Recommended: ['B018FK66TU', 'B00HY7VU60', 'B006MW3UZW', 'B00RT7K67E', 'B01DEBC7Q6', 'B00G5G7EXY', 'B00N1JQ452', 'B01BLS9E2Y', 'B00OV3VGP0', 'B00N1JQ2UO']


<span style="color:#66BB6A">**Observation:** across the sample users checked, the recommended items overlap 
heavily with each other (e.g. `B018FK66TU`, `B01DEBC7Q6`, `B008JFUPK8` appear for 
multiple users), even though what each user rated in `train` is completely different. 
This suggests the model is still leaning toward generally popular items rather than 
producing strongly personalized recommendations, likely because most users have very 
few train interactions giving ALS too little signal to learn a distinct taste vector per user. 
This matches the "sparse user" risk flagged in the Pre-Code Notes, and will be worth revisiting during 
tuning and evaluation.</span>

##### <span style="color:#BA68C8">Saving the recommendations</span>

We save the per-user top-10 ALS recommendations to `datasets/` as a parquet file, 
so we don't need to recompute them later during the Evaluation step (NDCG, 
Recall, Coverage).

In [13]:
user_ids_all = user_cat.cat.categories[all_user_indices]
recommended_asins_all = [item_cat.cat.categories[row].tolist() for row in all_recommended_items]

als_recommendations_df = pd.DataFrame({
    'user_id': user_ids_all,
    'top10': recommended_asins_all
})

als_recommendations_df.to_parquet('datasets/als_recommendations.parquet', index=False)

print("Saved ✅")
print(als_recommendations_df.head())

Saved ✅
                        user_id  \
0  AE2222FRPDMNOMYOMCWIANTXP7UQ   
1  AE22236AFRRSMQIKGG7TPTB75QEA   
2  AE222H3FGXWLHRFUMGMS2RR57NDQ   
3  AE223AAG2OHFGX6BTTD3HGJZCMDQ   
4  AE223CNRS5AQOKRWCFPRBUNJ5T2Q   

                                               top10  
0  [B018FK66TU, B01DEBC7Q6, B00N1JQ2UO, B01BLS9E2...  
1  [B018FK66TU, B00005JM5E, B01DEBC7Q6, B008JFUPK...  
2  [B018FK66TU, B00HY7VU60, B006MW3UZW, B00RT7K67...  
3  [B008BQ8YHQ, B003VPK1DW, B006MYGL8S, B008Y2X78...  
4  [B018FK66TU, B01DEBC7Q6, B00N1JQ2UO, B01BLS9E2...  
